# Phase 1 — Decontaminated Re-Scoring of the Selected Checkpoint

**No training or inference happens here.** The selected checkpoint of the Phase 2 matrix
(the best-validation Legal-BERT dual-head run)
already persisted its test logits to `runs/<run_id>/test_logits.npz` on the frozen seed-42
split. This notebook loads those logits and scores them twice:

| Set | Definition |
| --- | --- |
| **Full test** | all test clauses |
| **Decontaminated test** | test clauses whose nearest training clause has TF-IDF char n-gram cosine similarity **< 0.90** |

Consumer ToS are heavily templated, so a test clause that differs from a training clause
only in the service name is effectively memorised rather than understood. Reporting only
the full-test number overstates generalisation; reporting only the decontaminated number
discards legitimately recurring boilerplate. Both are reported side by side.

Every metric also gets a **percentile bootstrap 95% CI** over 1,000 clause-level
resamples, so the full-vs-decontaminated gap can be read against sampling noise.

Prerequisite: notebook `02_multiseed_encoder_runs.ipynb` has been executed, so
`generated_files/lawgic_taxonomy/runs*/` contains the run directories. Nothing here needs
a GPU, a checkpoint download, credentials, or network access.

Outputs (written to `generated_files/lawgic_taxonomy/evaluation*/`):
- `phase1_decontaminated.csv`
- `phase1_decontaminated.tex` (booktabs)
- `phase1_contamination_flags.csv` (per-test-clause similarity + flag)


In [ ]:
import os
import sys
from pathlib import Path

# ── Corpus version: set BEFORE importing lawgic_eval_core ─────────────────────
os.environ["LAWGIC_CORPUS_VERSION"] = "v2"


def find_project_root(start: Path) -> Path:
    for sentinel in [
        "generated_files/lawgic_taxonomy/lawgic_multihead_wide_v2.csv",
        "generated_files/lawgic_taxonomy/lawgic_multihead_wide.csv",
    ]:
        for candidate in (start, *start.parents):
            if (candidate / sentinel).exists():
                return candidate
    raise FileNotFoundError("Run this notebook from inside the lawgic repository.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

import json

import numpy as np
import pandas as pd

import lawgic_eval_core as core

pd.set_option("display.width", 140)
print(f"Project root : {PROJECT_ROOT}")
print(f"Corpus ver   : {core._CORPUS_VERSION}")
print(f"Checkpoint   : {core.CHECKPOINT_DIR}")
print(f"Split file   : {core.SPLIT_PATH}")
print(f"Topics       : {core.NUM_LAWGIC_TOPICS}")

## 1. Persist and load the seed-42 split

The fine-tuning notebook built its split **inline** and never wrote it to disk, so there
was no artifact any later analysis could point at. `core.persist_split()` re-runs that
exact logic — composite `topic__harmN` stratification key, rare keys (<5) collapsed to
`__rare__`, two-stage `train_test_split(random_state=42)`, exact-text leakage assertion —
and writes the assignment to a `splits/` CSV.

The cell below asserts the recovered row counts equal those recorded in
`core.EXPECTED_SPLIT_ROWS`. If that assertion ever fails, the corpus has changed
underneath the checkpoint and **no comparison in these notebooks is valid** — stop and
investigate rather than re-persisting.

In [ ]:
split_path = core.persist_split()          # no-op if the file already exists
corpus = core.load_corpus()
frames = core.split_frames(corpus)

counts = {name: len(frame) for name, frame in frames.items()}
assert counts == core.EXPECTED_SPLIT_ROWS, counts
print(f"Split artifact: {split_path}")
print(f"Row counts    : {counts}  (matches training_metadata.json)")

train_frame, test_frame = frames["train"], frames["test"]
test_arrays = core.label_arrays(test_frame)
print(f"Test observed topic cells: {int(test_arrays['label_masks'].sum()):,}")
print(f"Test rows with a risk label: {int(test_arrays['harm_masks'].sum()):,}")

## Load the persisted run logits

`RUN_ID` selects which Phase 2 run to re-score. The default is the deployment candidate:
the best-validation Legal-BERT dual-head seed, auto-detected from the runs directory.
The `.npz` stores logits, labels, masks and `row_id` in test-split order, so every
downstream number is recomputable on a laptop. The row-order assertion below guarantees
the logits line up with the split loaded above.


In [ ]:
import lawgic_train_matrix as tm

RUNS_DIR = tm.RUNS_DIR
LEGAL_BERT = tm.ENCODERS[0]

# Auto-detect best-validation Legal-BERT dual-head run from persisted metrics.
legal_bert_dual_runs = sorted(RUNS_DIR.glob("legal-bert-base-uncased__seed*__dual/metrics.json"))
if not legal_bert_dual_runs:
    raise FileNotFoundError(
        f"No Legal-BERT dual-head runs found in {RUNS_DIR}. "
        "Run 02_multiseed_encoder_runs.ipynb first."
    )
best_run = max(
    legal_bert_dual_runs,
    key=lambda p: json.loads(p.read_text())["best_val_metric"],
)
RUN_ID = best_run.parent.name
print(f"Selected run: {RUN_ID}")

payload = np.load(RUNS_DIR / RUN_ID / "test_logits.npz")

topic_logits, harm_logits = payload["topic_logits"], payload["harm_logits"]
assert np.array_equal(payload["row_id"], test_frame["row_id"].to_numpy()), (
    "run logits are not in the persisted test-split order"
)
for key, frame_key in (("labels", "labels"), ("label_masks", "label_masks"),
                       ("harm_labels", "harm_labels"), ("harm_masks", "harm_masks")):
    assert np.array_equal(payload[key], test_arrays[frame_key]), f"stored {key} drifted from the split"

print(f"topic_logits {topic_logits.shape} | harm_logits {harm_logits.shape}")

reference = core.all_metrics(topic_logits, harm_logits, test_arrays)
print("\nFull-test metrics recomputed from the stored logits:")
for key in core.HEADLINE_METRICS:
    print(f"  {key:16s} {reference[key]:.4f}")


### Sanity check against the run's recorded metrics

`runs/<run_id>/metrics.json` was written when the run trained. The values recomputed
above from the stored logits should match it to ~1e-4; a gap means the logits, the split,
or the metric code are not the ones the run used.


In [ ]:
recorded = json.loads((RUNS_DIR / RUN_ID / "metrics.json").read_text())

comparison = pd.DataFrame(
    [
        {"metric": m, "recorded": recorded[m], "recomputed": reference[m]}
        for m in core.HEADLINE_METRICS
    ]
)
comparison["abs_diff"] = (comparison["recorded"] - comparison["recomputed"]).abs()
display(comparison)

assert comparison["abs_diff"].max() < 1e-3, "recomputed metrics drift from the recorded run; investigate before reporting"


## 2. Flag near-duplicate contamination

Same vectoriser as `scripts/near_duplicate_split_audit.py`: TF-IDF over `char_wb`
3–5-grams with `min_df=2`, nearest-neighbour cosine against the **training** split only
(validation is irrelevant here — the question is what the model saw during fitting).

A test clause is *contaminated* at threshold `t` when its maximum similarity to any
training clause is `>= t`. The headline decontaminated set drops everything at `>= 0.90`;
the 0.80 and 0.95 columns are reported for context because the choice of threshold is a
judgement call, not a fact.

In [ ]:
similarity = core.max_train_similarity(
    train_frame[core.TEXT_COLUMN].astype(str).tolist(),
    test_frame[core.TEXT_COLUMN].astype(str).tolist(),
)

flags = pd.DataFrame(
    {
        "row_id": test_frame["row_id"].to_numpy(),
        "max_train_similarity": similarity,
        "contaminated_080": similarity >= 0.80,
        "contaminated_090": similarity >= core.CONTAMINATION_THRESHOLD,
        "contaminated_095": similarity >= 0.95,
        "text": test_frame[core.TEXT_COLUMN].astype(str).to_numpy(),
    }
)
core.EVAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
assert np.array_equal(flags["row_id"].to_numpy(), payload["row_id"])
flags.to_csv(core.EVAL_OUT_DIR / "phase1_contamination_flags.csv", index=False)

for threshold in (0.80, 0.90, 0.95):
    share = float((similarity >= threshold).mean())
    print(f"  test clauses with a training neighbour >= {threshold:.2f}: {share:.4f} ({int(share * len(similarity))} rows)")

clean_index = np.flatnonzero(similarity < core.CONTAMINATION_THRESHOLD)
full_index = np.arange(len(similarity))
print(f"\nDecontaminated subset: {len(clean_index)} of {len(full_index)} test clauses "
      f"({len(clean_index) / len(full_index):.1%} retained)")

## 3. Bootstrap 95% CIs on both sets

1,000 resamples, clauses drawn with replacement, percentile interval. The *clause* is the
resampling unit for both heads: topic cells within a clause are not independent (they
share the supervision mask and the encoder representation), so resampling cells would
understate the interval.

`point` is the metric on the observed data; `mean` is the bootstrap mean. A visible gap
between them signals a skewed sampling distribution — worth a footnote in the manuscript
if it appears.

In [ ]:
N_RESAMPLES = 1000

boot_full = core.bootstrap_ci(topic_logits, harm_logits, test_arrays, full_index, n_resamples=N_RESAMPLES)
boot_clean = core.bootstrap_ci(topic_logits, harm_logits, test_arrays, clean_index, n_resamples=N_RESAMPLES)

boot_full["set"] = "full"
boot_clean["set"] = "decontaminated"
display(pd.concat([boot_full, boot_clean], ignore_index=True))

## 4. Side-by-side table (CSV + LaTeX)

Rows = metric, columns = full vs decontaminated, cells = `point [CI-low, CI-high]`.
The `delta` column is decontaminated minus full: negative means the reported score was
inflated by memorised near-duplicates.

In [ ]:
LABELS = {
    "topic_macro_f1": "Topic macro-F1",
    "topic_micro_f1": "Topic micro-F1",
    "risk_accuracy": "Risk accuracy",
    "risk_macro_f1": "Risk macro-F1",
}


def cell(row: pd.Series) -> str:
    return f"{row['point']:.3f} [{row['ci_low']:.3f}, {row['ci_high']:.3f}]"


table = pd.DataFrame(
    [
        {
            "Metric": LABELS[metric],
            f"Full test (n={len(full_index)})": cell(boot_full.set_index("metric").loc[metric]),
            f"Decontaminated (n={len(clean_index)})": cell(boot_clean.set_index("metric").loc[metric]),
            "Delta": f"{boot_clean.set_index('metric').loc[metric, 'point'] - boot_full.set_index('metric').loc[metric, 'point']:+.3f}",
        }
        for metric in core.HEADLINE_METRICS
    ]
)
display(table)

csv_path, tex_path = core.write_outputs(
    table,
    "phase1_decontaminated",
    caption=(
        "Test performance of the selected dual-head Legal-BERT checkpoint (best-validation "
        "seed of the Phase 2 matrix) on the full test split "
        f"and on the decontaminated subset (test clauses whose nearest training clause has "
        f"TF-IDF char n-gram cosine similarity below {core.CONTAMINATION_THRESHOLD:.2f}). "
        f"Cells are point estimates with percentile bootstrap 95\\% confidence intervals "
        f"over {N_RESAMPLES} clause-level resamples."
    ),
    label="tab:decontaminated-eval",
)
print(csv_path)
print(tex_path)

## 5. Document-grouped re-split — **code only, not executed**

Random clause-level splitting is what lets near-duplicates straddle the train/test
boundary in the first place: two clauses from the same document (or two documents from
the same service, which often share a template) can land on opposite sides. A
document-grouped split assigns *every clause of one document to one split*, which removes
the mechanism rather than filtering its symptoms after the fact.

**Where the identifiers come from.** All three sources carry usable provenance in
`native_annotations`:

| Source | `source_id` format | Document key |
| --- | --- | --- |
| CLAUDETTE | `"23andme:36:use2"` (service : clause index : tag) | leading service name |
| 100 ToS | `"Badoo:5"` (platform : clause index) | leading platform name |
| ToS;DR | `"15447"` (bare point id) | `service_id` via join on `datasets/tos_dr/points.csv` |

ToS;DR points are annotations of a *service*, not of a single document, so the grouping
unit there is the service — the coarser and therefore safer choice: it prevents two points
quoting the same policy from splitting apart.

**Coverage.** The cell below prints the fraction of rows with no usable identifier. On the
current corpus this is **0%** (all rows resolve into ~2,000+ document/service groups)
provided `datasets/tos_dr/points.csv` is present — that file is gitignored, so on
a machine without the raw ToS;DR export the ToS;DR rows fall back to `None` and the cell
will report ~83% unidentified. Any row that cannot be resolved would be given a synthetic
singleton group (`unknown:<row_id>`), i.e. treated as its own document, which is the
conservative choice: it never merges unrelated clauses into one group.

**Why it is not executed.** Producing a second split would fork the corpus from the
trained checkpoint, and every number in Phases 1–3 is defined against the persisted
seed-42 assignment. Running this requires a full retrain to mean anything. Set
`RUN_DOCUMENT_SPLIT = True` and re-run only when you are ready to retrain against it.

In [ ]:
RUN_DOCUMENT_SPLIT = False  # deliberately off; see the markdown cell above

from sklearn.model_selection import GroupShuffleSplit


def document_groups(frame: pd.DataFrame) -> pd.Series:
    """Assign each clause a document/service group id from its provenance."""
    tosdr_map = core.tosdr_service_map()

    def resolve(row):
        direct = core.document_identifier(row["native_annotations"], row["sources"])
        if direct is not None:
            return direct
        for annotation in row["native_annotations"] or []:
            if annotation.get("source_dataset") == "tos_dr":
                mapped = tosdr_map.get(str(annotation.get("source_id")))
                if mapped is not None:
                    return mapped
        return None

    resolved = frame.apply(resolve, axis=1)
    unresolved = resolved.isna()
    print(f"Rows with no usable document identifier: {unresolved.mean():.4f} ({int(unresolved.sum())} rows)")
    # Conservative fallback: each unidentified row becomes its own group.
    return resolved.fillna(pd.Series("unknown:" + frame["row_id"].astype(str), index=frame.index))


def grouped_split(frame: pd.DataFrame, groups: pd.Series, seed: int = core.SEED) -> pd.Series:
    """80/10/10 split where no document appears in more than one split."""
    outer = GroupShuffleSplit(n_splits=1, test_size=core.VAL_SIZE + core.TEST_SIZE, random_state=seed)
    train_pos, holdout_pos = next(outer.split(frame, groups=groups))

    holdout = frame.iloc[holdout_pos]
    inner = GroupShuffleSplit(
        n_splits=1, test_size=core.TEST_SIZE / (core.VAL_SIZE + core.TEST_SIZE), random_state=seed
    )
    val_pos, test_pos = next(inner.split(holdout, groups=groups.iloc[holdout_pos]))

    assignment = pd.Series("train", index=frame.index, dtype=object)
    assignment.iloc[holdout_pos[val_pos]] = "validation"
    assignment.iloc[holdout_pos[test_pos]] = "test"
    return assignment


if RUN_DOCUMENT_SPLIT:
    groups = document_groups(corpus)
    print(f"Distinct document/service groups: {groups.nunique():,}")

    assignment = grouped_split(corpus, groups)
    print(assignment.value_counts().to_dict())

    # No group may straddle splits.
    straddling = groups.groupby(groups).apply(lambda g: assignment.loc[g.index].nunique()).gt(1).sum()
    assert straddling == 0, f"{straddling} groups appear in more than one split"

    out_path = core.SPLIT_DIR / "split_document_grouped_seed42.csv"
    pd.DataFrame({"row_id": corpus["row_id"], "split": assignment, "document_group": groups}).to_csv(
        out_path, index=False
    )
    print(f"Wrote {out_path} — retraining is required before any metric from it is comparable.")
else:
    print("RUN_DOCUMENT_SPLIT is False: no second split was produced (by design).")